# Features Transacciones VPC - TABLA ANCHA (cash-in + cash-out + transferencias salientes)
Tabla `HM_TRANSACCIONES_VPC` (panel RUC x periodo).
- **CASH-IN**: POS, DEPOSITOS, PAGOS RECIBIDOS, TRANSFERENCIAS
- **CASH-OUT**: PAGO DE SERVICIOS, PAGOS MASIVOS, **TRANSFERENCIAS SALIENTES** (`transf_out`: IBK + interbancarias + exterior)

Por n3 (monto): M0, m1, var1, varpct1, sum3/mm3, mm6, tend3, tend6. Trxs: M0, m1, var1. Por n2: monto, trxs.
Totales por flujo (cashin/cashout), **flujo_neto**, **ratio_cashin_cashout**, y conteo de flujos/tipos distintos (`n_flujos_cashin`, `n_flujos_cashout`, `n_tipos_mov`...). meses_activos_3m/6m.

In [ ]:
# 1) Instalacion
%pip install -q awswrangler

In [ ]:
# 2) Imports y CONFIG
import time
import awswrangler as wr

DATABASE       = "disc_comercial"
WORKGROUP      = "ibk-discovery-comercial-work-group"
S3_OUTPUT      = "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/athena-results/"
PERIODO_INICIO = 202401
PERIODO_FIN    = 202605

In [ ]:
# 3) Utilidades
def run_ddl(sql: str, label: str = ""):
    qid = wr.athena.start_query_execution(sql=sql, database=DATABASE, workgroup=WORKGROUP, s3_output=S3_OUTPUT)
    res = wr.athena.wait_query(query_execution_id=qid)
    state = res["Status"]["State"]
    if state != "SUCCEEDED":
        raise RuntimeError("[" + label + "] " + state + ": " + str(res["Status"].get("StateChangeReason","")))
    print("  OK [" + label + "] qid=" + qid)
    return qid

def drop_table(name: str):
    run_ddl("DROP TABLE IF EXISTS " + DATABASE + "." + name, "drop " + name)

In [ ]:
# 4) Template SQL
SQL_TXRS = r"""
-- ============================================================================
-- FEATURES TRANSACCIONES VPC - TABLA ANCHA (cada CASE con su CASH-IN/CASH-OUT explicito)
--   CASH-IN : POS, DEPOSITOS, PAGOS RECIBIDOS, TRANSFERENCIAS
--   CASH-OUT: PAGO DE SERVICIOS, PAGOS MASIVOS, TRANSFERENCIAS salientes (transf_out)
-- Fuente : e_perm_aws.t_agg_mes_vpc_transacc_sav_imp | Cruce: T_VPC_CLIENTE_BANCA_FINAL_HST
-- ============================================================================
-- DROP TABLE IF EXISTS disc_comercial.HM_TRANSACCIONES_VPC

CREATE TABLE disc_comercial.HM_TRANSACCIONES_VPC
WITH (format='Parquet', parquet_compression='SNAPPY', partitioned_by=ARRAY['periodo']) AS (

WITH params AS (
    SELECT {periodo_inicio} AS periodo_inicio, {periodo_fin} AS periodo_fin
),
periodos AS (
    SELECT periodo_val FROM (
        SELECT (anio*100 + mes) AS periodo_val
        FROM (SELECT anio FROM UNNEST(SEQUENCE(2024, 2026)) AS t(anio)) a
        CROSS JOIN (SELECT mes FROM UNNEST(SEQUENCE(1, 12)) AS t(mes)) m
    ) g CROSS JOIN params p
    WHERE g.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin
),
cuc_ruc AS (
    SELECT periodo_val, num_ruc_val, cuc_num
    FROM e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST
    CROSS JOIN params
    WHERE fecha_dt IN (
        SELECT MAX(fecha_dt) FROM e_perm_aws.T_VPC_CLIENTE_BANCA_FINAL_HST
        CROSS JOIN params p2
        WHERE CAST(YEAR(fecha_dt)*100 + MONTH(fecha_dt) AS INTEGER) BETWEEN p2.periodo_inicio AND p2.periodo_fin
        GROUP BY CAST(YEAR(fecha_dt)*100 + MONTH(fecha_dt) AS INTEGER)
    )
    AND COALESCE(num_ruc_val,'.') NOT LIKE '.' AND COALESCE(num_ruc_val,'.') NOT LIKE ''
),
universo_ruc AS (
    SELECT DISTINCT r.num_ruc_val
    FROM e_perm_aws.t_agg_mes_vpc_transacc_sav_imp t
    INNER JOIN cuc_ruc r ON t.cuc_num_val = r.cuc_num AND t.periodo_val = r.periodo_val
    CROSS JOIN params p
    WHERE t.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin AND (
          ( t.grupo_final_vpc_n3 IN ('POS','DEPOSITOS','PAGOS RECIBIDOS','TRANSFERENCIAS') AND t.tipo_cash_desc='CASH-IN'  )
       OR ( t.grupo_final_vpc_n3 IN ('PAGO DE SERVICIOS','PAGOS MASIVOS')                  AND t.tipo_cash_desc='CASH-OUT' )
       OR ( t.grupo_final_vpc_n3 = 'TRANSFERENCIAS'                                        AND t.tipo_cash_desc='CASH-OUT' )
      ) AND r.num_ruc_val IS NOT NULL
),
malla AS ( SELECT u.num_ruc_val, pe.periodo_val FROM universo_ruc u CROSS JOIN periodos pe ),
base AS (
    SELECT t.periodo_val, r.num_ruc_val,
           t.grupo_final_vpc_n3 AS n3, t.grupo_final_vpc_n2 AS n2, t.grupo_final_vpc_desc AS desc4,
           t.tipo_cash_desc AS flujo, t.cant_transacc, t.montotransaccion_solar_amt AS monto
    FROM e_perm_aws.t_agg_mes_vpc_transacc_sav_imp t
    INNER JOIN cuc_ruc r ON t.cuc_num_val = r.cuc_num AND t.periodo_val = r.periodo_val
    CROSS JOIN params p
    WHERE t.periodo_val BETWEEN p.periodo_inicio AND p.periodo_fin AND (
          ( t.grupo_final_vpc_n3 IN ('POS','DEPOSITOS','PAGOS RECIBIDOS','TRANSFERENCIAS') AND t.tipo_cash_desc='CASH-IN'  )
       OR ( t.grupo_final_vpc_n3 IN ('PAGO DE SERVICIOS','PAGOS MASIVOS')                  AND t.tipo_cash_desc='CASH-OUT' )
       OR ( t.grupo_final_vpc_n3 = 'TRANSFERENCIAS'                                        AND t.tipo_cash_desc='CASH-OUT' )
      ) AND r.num_ruc_val IS NOT NULL
),
agg AS (
    SELECT periodo_val, num_ruc_val,
        -- N3 (con su tipo CASH-IN / CASH-OUT explicito),
        SUM(CASE WHEN n3='POS' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS pos_monto,
        SUM(CASE WHEN n3='POS' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS pos_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS depos_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS depos_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS pag_recib_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS pag_recib_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS transf_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS transf_trxs,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS pag_serv_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS pag_serv_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS pag_masiv_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS pag_masiv_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS transf_out_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS transf_out_trxs,
        -- N2 (con su tipo explicito),
        SUM(CASE WHEN n3='POS' AND n2='VISA' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS pos_visa_monto,
        SUM(CASE WHEN n3='POS' AND n2='VISA' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS pos_visa_trxs,
        SUM(CASE WHEN n3='POS' AND n2='MASTERCARD' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS pos_master_monto,
        SUM(CASE WHEN n3='POS' AND n2='MASTERCARD' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS pos_master_trxs,
        SUM(CASE WHEN n3='POS' AND n2='AMEX' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS pos_amex_monto,
        SUM(CASE WHEN n3='POS' AND n2='AMEX' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS pos_amex_trxs,
        SUM(CASE WHEN n3='POS' AND n2='DINERS' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS pos_diners_monto,
        SUM(CASE WHEN n3='POS' AND n2='DINERS' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS pos_diners_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='ABONO POR CIERRE' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS depos_abono_cierre_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='ABONO POR CIERRE' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS depos_abono_cierre_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='REGULARIZACIONES ATM' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS depos_regul_atm_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='REGULARIZACIONES ATM' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS depos_regul_atm_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='CHEQUE' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS depos_cheque_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='CHEQUE' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS depos_cheque_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='EFECTIVO' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS depos_efectivo_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='EFECTIVO' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS depos_efectivo_trxs,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='REMESAS' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS depos_remesas_monto,
        SUM(CASE WHEN n3='DEPOSITOS' AND n2='REMESAS' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS depos_remesas_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='PROVEEDORES' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS pag_recib_prov_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='PROVEEDORES' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS pag_recib_prov_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='PLANILLAS' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS pag_recib_plan_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='PLANILLAS' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS pag_recib_plan_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='VARIOS' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS pag_recib_varios_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='VARIOS' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS pag_recib_varios_trxs,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='CTS' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS pag_recib_cts_monto,
        SUM(CASE WHEN n3='PAGOS RECIBIDOS' AND n2='CTS' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS pag_recib_cts_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA IBK' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS transf_ibk_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA IBK' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS transf_ibk_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANFERENCIA EXTERIOR' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS transf_exterior_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANFERENCIA EXTERIOR' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS transf_exterior_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA INTERBANCARIA' AND flujo='CASH-IN' THEN monto ELSE 0 END)         AS transf_interbanc_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA INTERBANCARIA' AND flujo='CASH-IN' THEN cant_transacc ELSE 0 END) AS transf_interbanc_trxs,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO SUNAT' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS pag_serv_sunat_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO SUNAT' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS pag_serv_sunat_trxs,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO AFP' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS pag_serv_afp_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO AFP' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS pag_serv_afp_trxs,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO DE SERVICIOS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS pag_serv_gen_monto,
        SUM(CASE WHEN n3='PAGO DE SERVICIOS' AND n2='PAGO DE SERVICIOS' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS pag_serv_gen_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='PROVEEDORES' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS pag_masiv_prov_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='PROVEEDORES' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS pag_masiv_prov_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='PLANILLAS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS pag_masiv_plan_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='PLANILLAS' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS pag_masiv_plan_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='CTS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS pag_masiv_cts_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='CTS' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS pag_masiv_cts_trxs,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='VARIOS' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS pag_masiv_varios_monto,
        SUM(CASE WHEN n3='PAGOS MASIVOS' AND n2='VARIOS' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS pag_masiv_varios_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA IBK' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS transf_out_ibk_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA IBK' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS transf_out_ibk_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA INTERBANCARIA' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS transf_out_interbanc_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANSFERENCIA INTERBANCARIA' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS transf_out_interbanc_trxs,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANFERENCIA EXTERIOR' AND flujo='CASH-OUT' THEN monto ELSE 0 END)         AS transf_out_exterior_monto,
        SUM(CASE WHEN n3='TRANSFERENCIAS' AND n2='TRANFERENCIA EXTERIOR' AND flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS transf_out_exterior_trxs,
        -- totales por flujo + diversidad,
        SUM(CASE WHEN flujo='CASH-IN'  THEN monto ELSE 0 END)         AS cashin_monto,
        SUM(CASE WHEN flujo='CASH-OUT' THEN monto ELSE 0 END)         AS cashout_monto,
        SUM(CASE WHEN flujo='CASH-IN'  THEN cant_transacc ELSE 0 END) AS cashin_trxs,
        SUM(CASE WHEN flujo='CASH-OUT' THEN cant_transacc ELSE 0 END) AS cashout_trxs,
        SUM(monto)            AS tot_monto,
        SUM(cant_transacc)    AS tot_trxs,
        COUNT(DISTINCT CASE WHEN flujo='CASH-IN'  THEN n3 END)    AS n_flujos_cashin,
        COUNT(DISTINCT CASE WHEN flujo='CASH-OUT' THEN n3 END)    AS n_flujos_cashout,
        COUNT(DISTINCT n3)    AS n_categorias_mov,
        COUNT(DISTINCT n2)    AS n_subcategorias_mov,
        COUNT(DISTINCT desc4) AS n_tipos_mov
    FROM base GROUP BY periodo_val, num_ruc_val
),
pivot AS (
    SELECT m.num_ruc_val, m.periodo_val,
        COALESCE(a.pos_monto,0) AS pos_monto,
        COALESCE(a.pos_trxs,0) AS pos_trxs,
        COALESCE(a.depos_monto,0) AS depos_monto,
        COALESCE(a.depos_trxs,0) AS depos_trxs,
        COALESCE(a.pag_recib_monto,0) AS pag_recib_monto,
        COALESCE(a.pag_recib_trxs,0) AS pag_recib_trxs,
        COALESCE(a.transf_monto,0) AS transf_monto,
        COALESCE(a.transf_trxs,0) AS transf_trxs,
        COALESCE(a.pag_serv_monto,0) AS pag_serv_monto,
        COALESCE(a.pag_serv_trxs,0) AS pag_serv_trxs,
        COALESCE(a.pag_masiv_monto,0) AS pag_masiv_monto,
        COALESCE(a.pag_masiv_trxs,0) AS pag_masiv_trxs,
        COALESCE(a.transf_out_monto,0) AS transf_out_monto,
        COALESCE(a.transf_out_trxs,0) AS transf_out_trxs,
        COALESCE(a.pos_visa_monto,0) AS pos_visa_monto,
        COALESCE(a.pos_visa_trxs,0) AS pos_visa_trxs,
        COALESCE(a.pos_master_monto,0) AS pos_master_monto,
        COALESCE(a.pos_master_trxs,0) AS pos_master_trxs,
        COALESCE(a.pos_amex_monto,0) AS pos_amex_monto,
        COALESCE(a.pos_amex_trxs,0) AS pos_amex_trxs,
        COALESCE(a.pos_diners_monto,0) AS pos_diners_monto,
        COALESCE(a.pos_diners_trxs,0) AS pos_diners_trxs,
        COALESCE(a.depos_abono_cierre_monto,0) AS depos_abono_cierre_monto,
        COALESCE(a.depos_abono_cierre_trxs,0) AS depos_abono_cierre_trxs,
        COALESCE(a.depos_regul_atm_monto,0) AS depos_regul_atm_monto,
        COALESCE(a.depos_regul_atm_trxs,0) AS depos_regul_atm_trxs,
        COALESCE(a.depos_cheque_monto,0) AS depos_cheque_monto,
        COALESCE(a.depos_cheque_trxs,0) AS depos_cheque_trxs,
        COALESCE(a.depos_efectivo_monto,0) AS depos_efectivo_monto,
        COALESCE(a.depos_efectivo_trxs,0) AS depos_efectivo_trxs,
        COALESCE(a.depos_remesas_monto,0) AS depos_remesas_monto,
        COALESCE(a.depos_remesas_trxs,0) AS depos_remesas_trxs,
        COALESCE(a.pag_recib_prov_monto,0) AS pag_recib_prov_monto,
        COALESCE(a.pag_recib_prov_trxs,0) AS pag_recib_prov_trxs,
        COALESCE(a.pag_recib_plan_monto,0) AS pag_recib_plan_monto,
        COALESCE(a.pag_recib_plan_trxs,0) AS pag_recib_plan_trxs,
        COALESCE(a.pag_recib_varios_monto,0) AS pag_recib_varios_monto,
        COALESCE(a.pag_recib_varios_trxs,0) AS pag_recib_varios_trxs,
        COALESCE(a.pag_recib_cts_monto,0) AS pag_recib_cts_monto,
        COALESCE(a.pag_recib_cts_trxs,0) AS pag_recib_cts_trxs,
        COALESCE(a.transf_ibk_monto,0) AS transf_ibk_monto,
        COALESCE(a.transf_ibk_trxs,0) AS transf_ibk_trxs,
        COALESCE(a.transf_exterior_monto,0) AS transf_exterior_monto,
        COALESCE(a.transf_exterior_trxs,0) AS transf_exterior_trxs,
        COALESCE(a.transf_interbanc_monto,0) AS transf_interbanc_monto,
        COALESCE(a.transf_interbanc_trxs,0) AS transf_interbanc_trxs,
        COALESCE(a.pag_serv_sunat_monto,0) AS pag_serv_sunat_monto,
        COALESCE(a.pag_serv_sunat_trxs,0) AS pag_serv_sunat_trxs,
        COALESCE(a.pag_serv_afp_monto,0) AS pag_serv_afp_monto,
        COALESCE(a.pag_serv_afp_trxs,0) AS pag_serv_afp_trxs,
        COALESCE(a.pag_serv_gen_monto,0) AS pag_serv_gen_monto,
        COALESCE(a.pag_serv_gen_trxs,0) AS pag_serv_gen_trxs,
        COALESCE(a.pag_masiv_prov_monto,0) AS pag_masiv_prov_monto,
        COALESCE(a.pag_masiv_prov_trxs,0) AS pag_masiv_prov_trxs,
        COALESCE(a.pag_masiv_plan_monto,0) AS pag_masiv_plan_monto,
        COALESCE(a.pag_masiv_plan_trxs,0) AS pag_masiv_plan_trxs,
        COALESCE(a.pag_masiv_cts_monto,0) AS pag_masiv_cts_monto,
        COALESCE(a.pag_masiv_cts_trxs,0) AS pag_masiv_cts_trxs,
        COALESCE(a.pag_masiv_varios_monto,0) AS pag_masiv_varios_monto,
        COALESCE(a.pag_masiv_varios_trxs,0) AS pag_masiv_varios_trxs,
        COALESCE(a.transf_out_ibk_monto,0) AS transf_out_ibk_monto,
        COALESCE(a.transf_out_ibk_trxs,0) AS transf_out_ibk_trxs,
        COALESCE(a.transf_out_interbanc_monto,0) AS transf_out_interbanc_monto,
        COALESCE(a.transf_out_interbanc_trxs,0) AS transf_out_interbanc_trxs,
        COALESCE(a.transf_out_exterior_monto,0) AS transf_out_exterior_monto,
        COALESCE(a.transf_out_exterior_trxs,0) AS transf_out_exterior_trxs,
        COALESCE(a.cashin_monto,0) AS cashin_monto,
        COALESCE(a.cashout_monto,0) AS cashout_monto,
        COALESCE(a.cashin_trxs,0) AS cashin_trxs,
        COALESCE(a.cashout_trxs,0) AS cashout_trxs,
        COALESCE(a.tot_monto,0) AS tot_monto,
        COALESCE(a.tot_trxs,0) AS tot_trxs,
        COALESCE(a.n_flujos_cashin,0) AS n_flujos_cashin,
        COALESCE(a.n_flujos_cashout,0) AS n_flujos_cashout,
        COALESCE(a.n_categorias_mov,0) AS n_categorias_mov,
        COALESCE(a.n_subcategorias_mov,0) AS n_subcategorias_mov,
        COALESCE(a.n_tipos_mov,0) AS n_tipos_mov,
        COALESCE(a.cashin_monto,0) - COALESCE(a.cashout_monto,0)           AS flujo_neto,
        COALESCE(a.cashin_monto,0) / NULLIF(COALESCE(a.cashout_monto,0),0) AS ratio_cashin_cashout,
        CASE WHEN a.num_ruc_val IS NOT NULL THEN 1 ELSE 0 END AS flag_activo_mes
    FROM malla m
    LEFT JOIN agg a ON m.num_ruc_val=a.num_ruc_val AND m.periodo_val=a.periodo_val
),
panel AS (
    SELECT pivot.*,
        LAG(pos_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS pos_monto_m1,
        pos_monto - LAG(pos_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS pos_monto_var1,
        pos_monto / NULLIF(LAG(pos_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS pos_monto_varpct1,
        SUM(pos_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS pos_monto_sum3,
        AVG(CAST(pos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS pos_monto_mm3,
        AVG(CAST(pos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS pos_monto_mm6,
        pos_monto - AVG(CAST(pos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS pos_monto_tend3,
        pos_monto - AVG(CAST(pos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS pos_monto_tend6,
        LAG(depos_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS depos_monto_m1,
        depos_monto - LAG(depos_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS depos_monto_var1,
        depos_monto / NULLIF(LAG(depos_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS depos_monto_varpct1,
        SUM(depos_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS depos_monto_sum3,
        AVG(CAST(depos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS depos_monto_mm3,
        AVG(CAST(depos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS depos_monto_mm6,
        depos_monto - AVG(CAST(depos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS depos_monto_tend3,
        depos_monto - AVG(CAST(depos_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS depos_monto_tend6,
        LAG(pag_recib_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS pag_recib_monto_m1,
        pag_recib_monto - LAG(pag_recib_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS pag_recib_monto_var1,
        pag_recib_monto / NULLIF(LAG(pag_recib_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS pag_recib_monto_varpct1,
        SUM(pag_recib_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS pag_recib_monto_sum3,
        AVG(CAST(pag_recib_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS pag_recib_monto_mm3,
        AVG(CAST(pag_recib_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS pag_recib_monto_mm6,
        pag_recib_monto - AVG(CAST(pag_recib_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS pag_recib_monto_tend3,
        pag_recib_monto - AVG(CAST(pag_recib_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS pag_recib_monto_tend6,
        LAG(transf_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS transf_monto_m1,
        transf_monto - LAG(transf_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS transf_monto_var1,
        transf_monto / NULLIF(LAG(transf_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS transf_monto_varpct1,
        SUM(transf_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS transf_monto_sum3,
        AVG(CAST(transf_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS transf_monto_mm3,
        AVG(CAST(transf_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS transf_monto_mm6,
        transf_monto - AVG(CAST(transf_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS transf_monto_tend3,
        transf_monto - AVG(CAST(transf_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS transf_monto_tend6,
        LAG(pag_serv_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS pag_serv_monto_m1,
        pag_serv_monto - LAG(pag_serv_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS pag_serv_monto_var1,
        pag_serv_monto / NULLIF(LAG(pag_serv_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS pag_serv_monto_varpct1,
        SUM(pag_serv_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS pag_serv_monto_sum3,
        AVG(CAST(pag_serv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS pag_serv_monto_mm3,
        AVG(CAST(pag_serv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS pag_serv_monto_mm6,
        pag_serv_monto - AVG(CAST(pag_serv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS pag_serv_monto_tend3,
        pag_serv_monto - AVG(CAST(pag_serv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS pag_serv_monto_tend6,
        LAG(pag_masiv_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS pag_masiv_monto_m1,
        pag_masiv_monto - LAG(pag_masiv_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS pag_masiv_monto_var1,
        pag_masiv_monto / NULLIF(LAG(pag_masiv_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS pag_masiv_monto_varpct1,
        SUM(pag_masiv_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS pag_masiv_monto_sum3,
        AVG(CAST(pag_masiv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS pag_masiv_monto_mm3,
        AVG(CAST(pag_masiv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS pag_masiv_monto_mm6,
        pag_masiv_monto - AVG(CAST(pag_masiv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS pag_masiv_monto_tend3,
        pag_masiv_monto - AVG(CAST(pag_masiv_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS pag_masiv_monto_tend6,
        LAG(transf_out_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS transf_out_monto_m1,
        transf_out_monto - LAG(transf_out_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS transf_out_monto_var1,
        transf_out_monto / NULLIF(LAG(transf_out_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS transf_out_monto_varpct1,
        SUM(transf_out_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS transf_out_monto_sum3,
        AVG(CAST(transf_out_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS transf_out_monto_mm3,
        AVG(CAST(transf_out_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS transf_out_monto_mm6,
        transf_out_monto - AVG(CAST(transf_out_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS transf_out_monto_tend3,
        transf_out_monto - AVG(CAST(transf_out_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS transf_out_monto_tend6,
        LAG(pos_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS pos_trxs_m1,
        pos_trxs - LAG(pos_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS pos_trxs_var1,
        LAG(depos_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS depos_trxs_m1,
        depos_trxs - LAG(depos_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS depos_trxs_var1,
        LAG(pag_recib_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS pag_recib_trxs_m1,
        pag_recib_trxs - LAG(pag_recib_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS pag_recib_trxs_var1,
        LAG(transf_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS transf_trxs_m1,
        transf_trxs - LAG(transf_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS transf_trxs_var1,
        LAG(pag_serv_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS pag_serv_trxs_m1,
        pag_serv_trxs - LAG(pag_serv_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS pag_serv_trxs_var1,
        LAG(pag_masiv_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS pag_masiv_trxs_m1,
        pag_masiv_trxs - LAG(pag_masiv_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS pag_masiv_trxs_var1,
        LAG(transf_out_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS transf_out_trxs_m1,
        transf_out_trxs - LAG(transf_out_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS transf_out_trxs_var1,
        LAG(cashin_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS cashin_monto_m1,
        cashin_monto - LAG(cashin_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS cashin_monto_var1,
        cashin_monto / NULLIF(LAG(cashin_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS cashin_monto_varpct1,
        SUM(cashin_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS cashin_monto_sum3,
        AVG(CAST(cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS cashin_monto_mm3,
        AVG(CAST(cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS cashin_monto_mm6,
        cashin_monto - AVG(CAST(cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS cashin_monto_tend3,
        cashin_monto - AVG(CAST(cashin_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS cashin_monto_tend6,
        LAG(cashout_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS cashout_monto_m1,
        cashout_monto - LAG(cashout_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS cashout_monto_var1,
        cashout_monto / NULLIF(LAG(cashout_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS cashout_monto_varpct1,
        SUM(cashout_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS cashout_monto_sum3,
        AVG(CAST(cashout_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS cashout_monto_mm3,
        AVG(CAST(cashout_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS cashout_monto_mm6,
        cashout_monto - AVG(CAST(cashout_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS cashout_monto_tend3,
        cashout_monto - AVG(CAST(cashout_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS cashout_monto_tend6,
        LAG(tot_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS tot_monto_m1,
        tot_monto - LAG(tot_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS tot_monto_var1,
        tot_monto / NULLIF(LAG(tot_monto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS tot_monto_varpct1,
        SUM(tot_monto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS tot_monto_sum3,
        AVG(CAST(tot_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS tot_monto_mm3,
        AVG(CAST(tot_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS tot_monto_mm6,
        tot_monto - AVG(CAST(tot_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS tot_monto_tend3,
        tot_monto - AVG(CAST(tot_monto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS tot_monto_tend6,
        LAG(flujo_neto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                      AS flujo_neto_m1,
        flujo_neto - LAG(flujo_neto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)                AS flujo_neto_var1,
        flujo_neto / NULLIF(LAG(flujo_neto,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val),0) - 1  AS flujo_neto_varpct1,
        SUM(flujo_neto) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)                          AS flujo_neto_sum3,
        AVG(CAST(flujo_neto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)          AS flujo_neto_mm3,
        AVG(CAST(flujo_neto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)          AS flujo_neto_mm6,
        flujo_neto - AVG(CAST(flujo_neto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS flujo_neto_tend3,
        flujo_neto - AVG(CAST(flujo_neto AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS flujo_neto_tend6,
        LAG(cashin_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS cashin_trxs_m1,
        cashin_trxs - LAG(cashin_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS cashin_trxs_var1,
        LAG(cashout_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS cashout_trxs_m1,
        cashout_trxs - LAG(cashout_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS cashout_trxs_var1,
        LAG(tot_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS tot_trxs_m1,
        tot_trxs - LAG(tot_trxs,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS tot_trxs_var1,
        LAG(ratio_cashin_cashout,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS ratio_cashin_cashout_m1,
        ratio_cashin_cashout - LAG(ratio_cashin_cashout,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS ratio_cashin_cashout_var1,
        LAG(n_tipos_mov,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS n_tipos_mov_m1,
        n_tipos_mov - LAG(n_tipos_mov,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS n_tipos_mov_var1,
        AVG(CAST(n_tipos_mov AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS n_tipos_mov_mm3,
        LAG(n_flujos_cashin,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS n_flujos_cashin_m1,
        n_flujos_cashin - LAG(n_flujos_cashin,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS n_flujos_cashin_var1,
        AVG(CAST(n_flujos_cashin AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS n_flujos_cashin_mm3,
        LAG(n_flujos_cashout,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)        AS n_flujos_cashout_m1,
        n_flujos_cashout - LAG(n_flujos_cashout,1) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val)  AS n_flujos_cashout_var1,
        AVG(CAST(n_flujos_cashout AS double)) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) AS n_flujos_cashout_mm3,
        SUM(flag_activo_mes) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)    AS meses_activos_3m,
        SUM(flag_activo_mes) OVER (PARTITION BY num_ruc_val ORDER BY periodo_val ROWS BETWEEN 5 PRECEDING AND CURRENT ROW)    AS meses_activos_6m
    FROM pivot
)
SELECT p.*, CAST(p.periodo_val AS VARCHAR) AS periodo
FROM panel p
)
"""

## Ejecutar

In [ ]:
# 5) Crear la tabla
t0 = time.time()
drop_table("HM_TRANSACCIONES_VPC")
run_ddl(SQL_TXRS.format(periodo_inicio=PERIODO_INICIO, periodo_fin=PERIODO_FIN), "transacciones")
print("Listo en " + str(round((time.time()-t0)/60,1)) + " min. Tabla: " + DATABASE + ".HM_TRANSACCIONES_VPC")